<a href="https://colab.research.google.com/github/VHugoLopez/Machine-L/blob/main/Unidad3/U3Practica2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Importamos librerias necesarias para regresion logistica y evaluacion de modelos
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [13]:
prestamos_df = pd.read_csv('lending_club_2007_2011_6_states.csv')
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [27]:
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

In [28]:
# Dividimos el dataFrame df en df_train y df_test.
# 60% para dataset de entrenamient0, y 40% para dataset de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [29]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 10), (7964, 10), (11944,), (7964,))

In [31]:
# Identificamos e imputamos los valores nulos con la mediana de cada columna
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

# Creamoos el clasificador de regresion logistica
clf1 = LogisticRegression(random_state=0, max_iter=1000)
# Entrenamos el clasificador
clf1.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



LogisticRegression(max_iter=1000, random_state=0)

In [32]:
# verificamos la precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento",
clf1.score(X_train, y_train) )

Precision del clasificador en fase de entrenamiento 0.8557434695244475


In [33]:
# Realizar una prediccion con los datos de prueba
y_pred = clf1.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
# Coeficiente de determinación
print( "Precisión:", clf1.score(X_test, y_test) )


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.18      0.01      0.01      1220
      Pagado       0.85      1.00      0.92      6744

    accuracy                           0.84      7964
   macro avg       0.52      0.50      0.46      7964
weighted avg       0.75      0.84      0.78      7964

Precisión: 0.8437970868910095


In [34]:
matriz_sin_balanceo = confusion_matrix(y_test, y_pred )
print(f'Matriz Confusion:\n', matriz_sin_balanceo)

Matriz Confusion:
 [[   7 1213]
 [  31 6713]]


In [35]:
# Crear segundo clasificador con balanceo de clases
clf2 = LogisticRegression(random_state=0, class_weight="balanced")
# Entrenar el nuevo clasificador
clf2.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



LogisticRegression(class_weight='balanced', random_state=0)

In [36]:
# verificamos la precisión del modelo 2 en la fase de entrenamiento
print("Exactitud del modelo de entrenamiento:", clf2.score(X_train, y_train))

Exactitud del modelo de entrenamiento: 0.5476389819156061


In [37]:
# Realizar una prediccion con los datos de prueba
y_pred2 = clf2.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador: \n",
classification_report(y_test, y_pred2, target_names=["No Pagado", "Pagado"]))
# Coeficiente de determinación
print( "Precisión:", clf2.score(X_test, y_test) )


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.20      0.68      0.31      1220
      Pagado       0.90      0.51      0.65      6744

    accuracy                           0.54      7964
   macro avg       0.55      0.60      0.48      7964
weighted avg       0.79      0.54      0.60      7964

Precisión: 0.5364138623807132


In [38]:
matriz_balanceo = confusion_matrix(y_test, y_pred2 )
print(f'Matriz Confusion:\n', matriz_balanceo)

Matriz Confusion:
 [[ 835  385]
 [3307 3437]]


In [39]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clf2, X_test, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
"Variable": X.columns,
"Importancia Media": result.importances_mean,
"Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

               Variable  Importancia Media  Desviación
1              int_rate           0.036154    0.003373
6            annual_inc           0.025084    0.003415
2            grade_code           0.009309    0.002300
7                   dti           0.004269    0.002660
3          purpose_code           0.003608    0.001543
0           funded_amnt           0.002302    0.001644
5   home_ownership_code           0.002281    0.001275
8            revol_util           0.000866    0.001151
4       addr_state_code           0.000486    0.001162
9  pub_rec_bankruptcies          -0.000088    0.000098


In [41]:
# Visualizar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
             error_y="Desviación", title="Importancia de características (Regresión Logística - Permutación)",
             text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

In [43]:
from sklearn.preprocessing import StandardScaler
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
y = prestamos_df["repaid"]
# stratify es para que mantenga la misma proporción de clases en ambos conjuntos
# Usamos los mismos parametros de tamaño y random usados anteriormente para que sea justa la compara
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.4, random_state=42, stratify=y)
# Normalizar variables (Regresión Logística (RL) es sensible a la escala)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [45]:
# 1. Imputamos los valores nulos con la mediana antes de escalar
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

# 2. Escalamos las características nuevamente para evitar NaNs y errores de convergencia
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Entrenamos el modelo RL escalado y balanceado
RL_scaled = LogisticRegression(random_state=0, class_weight='balanced', max_iter=1000)
RL_scaled.fit(X_train_scaled, y_train)

# 4. Probar y evaluar el modelo RL escalado
y_pred_scaled = RL_scaled.predict(X_test_scaled)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))


Reporte de clasificación:
              precision    recall  f1-score   support

   No Pagado       0.22      0.61      0.32      1177
      Pagado       0.90      0.62      0.73      6787

    accuracy                           0.61      7964
   macro avg       0.56      0.61      0.53      7964
weighted avg       0.80      0.61      0.67      7964



In [46]:
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred_scaled))


Matriz de Confusión:
[[ 719  458]
 [2611 4176]]


In [47]:
from sklearn.inspection import permutation_importance
result = permutation_importance(RL_scaled, X_test_scaled, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
"Variable": X.columns,
"Importancia Media": result.importances_mean,
"Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

               Variable  Importancia Media  Desviación
1              int_rate           0.039160    0.004612
6            annual_inc           0.009066    0.002748
5   home_ownership_code           0.002712    0.001281
3          purpose_code           0.001666    0.002307
2            grade_code           0.000255    0.000289
4       addr_state_code          -0.000184    0.002106
7                   dti          -0.000209    0.000735
0           funded_amnt          -0.001285    0.002075
9  pub_rec_bankruptcies          -0.002017    0.000916
8            revol_util          -0.002436    0.001016


In [49]:
# Visualizar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
             error_y="Desviación", title="Importancia de características (Regresión Logística - Permutación)",
             text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

In [50]:
# Ajustar el umbral de decisión a 0.45
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.45
umbral = 0.45   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.45:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.45
              precision    recall  f1-score   support

   No Pagado       0.24      0.50      0.32      1177
      Pagado       0.89      0.72      0.80      6787

    accuracy                           0.69      7964
   macro avg       0.57      0.61      0.56      7964
weighted avg       0.80      0.69      0.73      7964


Matriz de Confusión con umbral de decisión a 0.45:
[[ 587  590]
 [1880 4907]]


In [51]:
# Ajustar el umbral de decisión a 0.40
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.40
umbral = 0.40   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.40:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.4
              precision    recall  f1-score   support

   No Pagado       0.26      0.37      0.30      1177
      Pagado       0.88      0.82      0.85      6787

    accuracy                           0.75      7964
   macro avg       0.57      0.59      0.58      7964
weighted avg       0.79      0.75      0.77      7964


Matriz de Confusión con umbral de decisión a 0.40:
[[ 433  744]
 [1251 5536]]


In [52]:
# Ajustar el umbral de decisión a 0.55
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.55
umbral = 0.55   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.55")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.55
              precision    recall  f1-score   support

   No Pagado       0.21      0.76      0.33      1177
      Pagado       0.92      0.50      0.65      6787

    accuracy                           0.54      7964
   macro avg       0.57      0.63      0.49      7964
weighted avg       0.82      0.54      0.60      7964


Matriz de Confusión con umbral de decisión a 0.55
[[ 890  287]
 [3389 3398]]


In [54]:
# Ajustar el umbral de decisión a 0.60
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.60
umbral = 0.60   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.60")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.6
              precision    recall  f1-score   support

   No Pagado       0.19      0.84      0.31      1177
      Pagado       0.93      0.38      0.54      6787

    accuracy                           0.45      7964
   macro avg       0.56      0.61      0.43      7964
weighted avg       0.82      0.45      0.51      7964


Matriz de Confusión con umbral de decisión a 0.60
[[ 988  189]
 [4204 2583]]


In [55]:
# Debemos proporcionar 10 valores (variables predictoras)
yhat_predmanual =  clf2.predict( [[5000, 15.5 , 5, 3, 4, 4, 30000, 12, 12, 0 ]] )

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LogisticRegression was fitted with feature names

